<a href="https://colab.research.google.com/github/yshzjq/Step-2.Statistical_Thinking-Machine-Learning/blob/main/Step%202.%20%ED%86%B5%EA%B3%84%EC%A0%81%20%EC%82%AC%EA%B3%A0%EC%99%80%20%EB%A8%B8%EC%8B%A0%EB%9F%AC%EB%8B%9D%20%EC%9E%85%EB%AC%B8/2_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 데이터 처리용
import pandas as pd# 데이터프레임 처리
import numpy as np# 수치 계산

# 시각화용
import matplotlib.pyplot as plt# 그래프
import seaborn as sns# 예쁜 그래프

# 머신러닝 / 텍스트 처리
from sklearn.model_selection import train_test_split# 데이터 분리
from sklearn.feature_extraction.text import CountVectorizer# BoW
from sklearn.feature_extraction.text import TfidfVectorizer# TF-IDF
from sklearn.naive_bayes import MultinomialNB # 나이브 베이즈
from sklearn.metrics import classification_report# 평가 리포트

In [3]:
# SMS 스팸 데이터 URL (공용 미러)
url ="https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

# TSV 파일 읽기
df = pd.read_csv(url, sep='\t', header=None)

# 컬럼명 지정
df.columns = ['label','message']

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
# 'ham' → 0, 'spam' → 1로 변환
df['label_num'] = df['label'].map({'ham':0,'spam':1})

df.head()

,label,message,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [5]:
# 입력(X): 문자 메시지
X = df['message']

# 정답(y): 스팸 여부
y = df['label_num']

# 데이터 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y# 스팸/정상 비율 유지
)

In [6]:
# Bag of Words 벡터라이저 생성
bow = CountVectorizer()

# 학습 데이터 기준으로 단어 사전 생성 + 변환
X_train_bow = bow.fit_transform(X_train)

# 테스트 데이터 변환
X_test_bow = bow.transform(X_test)

bow.vocabulary_

{'he': 3373,
 'will': 7439,
 'you': 7629,
 'guys': 3290,
 'close': 1861,
 'can': 1646,
 'please': 5210,
 'come': 1919,
 'up': 7107,
 'now': 4812,
 'imin': 3629,
 'town': 6927,
 'dontmatter': 2420,
 'if': 3608,
 'urgoin': 7133,
 'outl8r': 4981,
 'just': 3873,
 'reallyneed': 5558,
 '2docd': 390,
 'dontplease': 2421,
 'dontignore': 2419,
 'mycalls': 4645,
 'no': 4770,
 'thecd': 6748,
 'isv': 3759,
 'important': 3637,
 'tome': 6881,
 '2moro': 403,
 'ok': 4889,
 'sry': 6376,
 'knw': 3962,
 'siva': 6148,
 'tats': 6648,
 'askd': 1104,
 'll': 4157,
 'see': 5933,
 'but': 1592,
 'prolly': 5403,
 'yeah': 7601,
 'swing': 6595,
 'by': 1607,
 'in': 3648,
 'bit': 1380,
 'got': 3209,
 'some': 6250,
 'things': 6775,
 'to': 6859,
 'take': 6621,
 'care': 1667,
 'of': 4864,
 'here': 3420,
 'firsg': 2887,
 'shall': 6011,
 'book': 1440,
 'chez': 1788,
 'jules': 3864,
 'for': 2951,
 'half': 3310,
 'eight': 2559,
 'that': 6739,
 'with': 7470,
 'thanks': 6733,
 'your': 7633,
 'message': 4444,
 'really': 5557,


In [7]:
bow.get_feature_names_out()

array(['00', '000', '000pes', ..., 'èn', 'ú1', '〨ud'], dtype=object)

In [8]:
bow_df = pd.DataFrame(
    X_train_bow.toarray(),
    columns=bow.get_feature_names_out()
)

bow_df.head()

,00,000,000pes,008704050406,0089,0121,01223585236,01223585334,02,0207,...,zed,zeros,zhong,zindgi,zoe,zogtorius,zyada,èn,ú1,〨ud
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
row0 = bow_df.iloc[1]

# 값이 0이 아닌 컬럼 위치
col_positions = row0[row0 > 0].index
col_positions

Index(['2docd', '2moro', 'can', 'come', 'dontignore', 'dontmatter',
       'dontplease', 'if', 'imin', 'important', 'isv', 'just', 'mycalls', 'no',
       'now', 'outl8r', 'please', 'reallyneed', 'thecd', 'tome', 'town', 'up',
       'urgoin'],
      dtype='object')

In [10]:
print(df.shape)

print(df.tail())

(5572, 3)
     label                                            message  label_num
5567  spam  This is the 2nd time we have tried 2 contact u...          1
5568   ham               Will ü b going to esplanade fr home?          0
5569   ham  Pity, * was in mood for that. So...any other s...          0
5570   ham  The guy did some bitching but I acted like i'd...          0
5571   ham                         Rofl. Its true to its name          0


In [13]:
# 나이브 베이즈 모델 생성
nb_bow = MultinomialNB()

# 모델 학습
nb_bow.fit(X_train_bow, y_train)

MultinomialNB()

In [14]:
# 예측 수행
y_pred_bow = nb_bow.predict(X_test_bow)

# 분류 성능 출력
print("Bag of Words 결과")
print(classification_report(y_test, y_pred_bow))

Bag of Words 결과
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.99      0.92      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [15]:
# TF-IDF 벡터라이저 생성
tfidf = TfidfVectorizer()

# 학습 데이터 변환
X_train_tfidf = tfidf.fit_transform(X_train)

# 테스트 데이터 변환
X_test_tfidf = tfidf.transform(X_test)


In [20]:
tfidf.get_feature_names_out()

array(['00', '000', '000pes', ..., 'èn', 'ú1', '〨ud'], dtype=object)

In [21]:
tfidf_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df.head()

,00,000,000pes,008704050406,0089,0121,01223585236,01223585334,02,0207,...,zed,zeros,zhong,zindgi,zoe,zogtorius,zyada,èn,ú1,〨ud
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
row0 = tfidf_df.iloc[1]

# 값이 정확히 1인 컬럼 위치 (정수 인덱스)
col_positions = row0[row0 > 0].index
col_positions

Index(['2docd', '2moro', 'can', 'come', 'dontignore', 'dontmatter',
       'dontplease', 'if', 'imin', 'important', 'isv', 'just', 'mycalls', 'no',
       'now', 'outl8r', 'please', 'reallyneed', 'thecd', 'tome', 'town', 'up',
       'urgoin'],
      dtype='object')

In [23]:
# 나이브 베이즈 모델 생성
nb_tfidf = MultinomialNB()

# 모델 학습
nb_tfidf.fit(X_train_tfidf, y_train)

MultinomialNB()

In [24]:
# 예측 수행
y_pred_tfidf = nb_tfidf.predict(X_test_tfidf)

# 분류 성능 출력
print("TF-IDF 결과")
print(classification_report(y_test, y_pred_tfidf))


TF-IDF 결과
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       1.00      0.70      0.83       149

    accuracy                           0.96      1115
   macro avg       0.98      0.85      0.90      1115
weighted avg       0.96      0.96      0.96      1115



In [25]:
# 테스트 데이터의 스팸 확률 예측
spam_prob_bow = nb_bow.predict_proba(X_test_bow)

spam_prob_bow[:10]


array([[9.99995084e-01, 4.91581714e-06],
       [1.00000000e+00, 8.45581173e-17],
       [1.00000000e+00, 1.16633731e-10],
       [6.48593235e-11, 1.00000000e+00],
       [9.99989877e-01, 1.01226136e-05],
       [1.00000000e+00, 9.38066487e-17],
       [9.99999814e-01, 1.85659269e-07],
       [9.99999951e-01, 4.85891046e-08],
       [1.00000000e+00, 6.97120532e-24],
       [9.99999995e-01, 4.66633182e-09]])

In [26]:
# 테스트 데이터의 스팸 확률 예측
spam_prob_tfidf = nb_tfidf.predict_proba(X_test_tfidf)

spam_prob_tfidf[:10]

array([[9.97627805e-01, 2.37219550e-03],
       [9.99878815e-01, 1.21185248e-04],
       [9.99198236e-01, 8.01764388e-04],
       [7.86434728e-02, 9.21356527e-01],
       [9.94893289e-01, 5.10671079e-03],
       [9.99104766e-01, 8.95233618e-04],
       [9.96308304e-01, 3.69169574e-03],
       [9.96064699e-01, 3.93530124e-03],
       [9.99892705e-01, 1.07294709e-04],
       [9.94842247e-01, 5.15775297e-03]])